# UC00222: Project 35 - School Enrollment Trend Analysis

Dennis Jojo Kuriakose (S224727875)

Hans Devu Sunil (S224072841)


## Project focus

Our project looks at how school enrollment changes over time and how those changes may differ by school type, education sector, and region.

To do that, enrollment data from several years needs to be brought together and linked with school location information. Before any real comparison can be made, the data has to be checked carefully because the yearly files are not perfectly aligned and some school identifiers are not unique on their own.


In [1]:
import pandas as pd
import requests


## Load school location data

The first dataset used here contains school location and profile information.  
This includes details such as school name, sector, school type, and regional information, which will later help when comparing enrollment patterns across different groups of schools.


In [2]:
# Import school locations dataset from API
base_url = "https://discover.data.vic.gov.au/api/3/action/datastore_search"
resource_id = "a70cb615-ed28-48b6-bebe-2d0d346f7927"
url = f"{base_url}?resource_id={resource_id}&limit=5000"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    school_locations_df = pd.DataFrame(data["result"]["records"])
    print("School location dataset loaded successfully.")
    print(school_locations_df.shape)
    display(school_locations_df.head())
else:
    print(f"Request failed with status code {response.status_code}")


School location dataset loaded successfully.
(2294, 23)


,_id,Education_Sector,Entity_Type,School_No,School_Name,School_Type,Address_Line_1,Address_Line_2,Address_Town,Address_State,...,Postal_Town,Postal_State,Postal_Postcode,Full_Phone_No,DE_Admin_Region,DE_Admin_AREA,LGA_ID,LGA_Name,X,Y
0,1,Catholic,2,20,Parade College,Secondary,1436 Plenty Road,None,BUNDOORA,VIC,...,BUNDOORA,VIC,3083,03 9468 3300,NORTH-WESTERN VICTORIA,North Eastern Melbourne,66,Banyule (C),145.06,-37.69
1,2,Catholic,2,25,Simonds Catholic College,Secondary,273 Victoria Street,None,WEST MELBOURNE,VIC,...,WEST MELBOURNE,VIC,3003,03 9321 9200,SOUTH-WESTERN VICTORIA,Western Melbourne,460,Melbourne (C),144.95,-37.8
2,3,Catholic,2,26,St Mary’s College Melbourne,Secondary,11 Westbury Street,None,ST KILDA EAST,VIC,...,ST KILDA,VIC,3182,03 9529 6611,SOUTH-EASTERN VICTORIA,Bayside Peninsula,590,Port Phillip (C),144.99,-37.85
3,4,Catholic,2,28,St Patrick's College Ballarat,Secondary,1431 Sturt Street,None,BALLARAT,VIC,...,BALLARAT,VIC,3350,03 5331 1688,SOUTH-WESTERN VICTORIA,Central Highlands,57,Ballarat (C),143.83,-37.55
4,5,Catholic,2,29,St Patrick's School,Primary,119 Drummond Street South,None,BALLARAT,VIC,...,BALLARAT WEST,VIC,3350,03 5332 7680,SOUTH-WESTERN VICTORIA,Central Highlands,57,Ballarat (C),143.84,-37.56


## Load enrollment data for multiple years

Since the project is about trends rather than a single-year snapshot, the enrollment files for 2022, 2023, 2024, and 2025 are loaded together.

These files will be combined first and then checked for consistency before further analysis.


In [3]:
# Enrollment datasets for multiple years
files = {
    "2022": "https://raw.githubusercontent.com/Chameleon-company/MOP-Code/master/datascience/usecases/DEPENDENCIES/UC00206_Project%2035_Source%20Data/dv335-allschoolsFTEenrolmentsFeb2022.csv",
    "2023": "https://raw.githubusercontent.com/Chameleon-company/MOP-Code/master/datascience/usecases/DEPENDENCIES/UC00206_Project%2035_Source%20Data/dv355-VIC%20All%20Schools%20Enrolments%202023.csv",
    "2024": "https://raw.githubusercontent.com/Chameleon-company/MOP-Code/master/datascience/usecases/DEPENDENCIES/UC00206_Project%2035_Source%20Data/dv377_DataVic-AllSchoolsEnrolments-2024.csv",
    "2025": "https://raw.githubusercontent.com/Chameleon-company/MOP-Code/master/datascience/usecases/DEPENDENCIES/UC00206_Project%2035_Source%20Data/dv403-AllSchoolsEnrolments-2025.csv",
}


In [4]:
dfs = []

for year, url in files.items():
    df = pd.read_csv(url, encoding="latin1")

    # Clean column names
    df.columns = (
        df.columns
        .str.strip()
        .str.replace('"', '', regex=False)
        .str.replace(" ", "_")
    )

    # Add year label
    df["Census_Year"] = int(year)

    # Standardise School_No as string
    df["School_No"] = df["School_No"].astype(str).str.strip()

    dfs.append(df)

enrolments_all_df = pd.concat(dfs, ignore_index=True)

print("Combined enrollment dataset shape:")
print(enrolments_all_df.shape)
display(enrolments_all_df.head())


Combined enrollment dataset shape:
(9172, 34)


,Education_Sector,Entity_Type,School_No,School_Name,School_Type,School_Status,Prep_Total,Year_1_Total,Year_2_Total,Year_3_Total,...,Year,CENSUS_TYPE,Census_Year,DE_Admin_Region,LGA_Name,DE_Admin_AREA,ï»¿Education_Sector,Region_Name,AREA_Name,LGA_TYPE
0,Catholic,2,204,St Monica's School,Primary,O,47.0,34.0,58.0,55.0,...,2022,F,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Catholic,2,205,St Joseph's School,Primary,O,52.0,55.0,42.0,66.0,...,2022,F,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Catholic,2,212,St Brendan's Primary School,Primary,O,1.0,2.0,3.0,3.0,...,2022,F,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Catholic,2,219,Sacred Heart College,Secondary,O,0.0,0.0,0.0,0.0,...,2022,F,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Catholic,2,227,St Patrick's School,Primary,O,17.0,23.0,21.0,11.0,...,2022,F,2022,NaN,NaN,NaN,NaN,NaN,NaN,NaN


After combining the annual files, the next step is to make sure that the same kinds of variables are available across all years.

This is important because some yearly files may include slightly different columns or formatting.


## Compare column structure across years

Before using the combined enrollment data, we checked which columns are shared across all four yearly files.

Only the overlapping columns are kept so that later comparisons are based on the same set of variables throughout the dataset.


In [5]:
# Find common columns across all year datasets
column_sets = [set(df.columns) for df in dfs]
common_columns = sorted(list(set.intersection(*column_sets)))

print("Number of common columns across all years:", len(common_columns))
print(common_columns)


Number of common columns across all years: 26
['CENSUS_TYPE', 'Census_Year', 'Entity_Type', 'Grand_Total', 'Prep_Total', 'Primary_Total', 'Primary_Ungraded_Total', 'School_Name', 'School_No', 'School_Status', 'School_Type', 'Secondary_Total', 'Secondary_Ungraded_Total', 'Year', 'Year_10_Total', 'Year_11_Total', 'Year_12_Total', 'Year_1_Total', 'Year_2_Total', 'Year_3_Total', 'Year_4_Total', 'Year_5_Total', 'Year_6_Total', 'Year_7_Total', 'Year_8_Total', 'Year_9_Total']


In [6]:
# Keep only common columns
enrolments_all_df = pd.concat([df[common_columns] for df in dfs], ignore_index=True)

print("Enrollment dataset after keeping common columns:")
print(enrolments_all_df.shape)
display(enrolments_all_df.head())


Enrollment dataset after keeping common columns:
(9172, 26)


,CENSUS_TYPE,Census_Year,Entity_Type,Grand_Total,Prep_Total,Primary_Total,Primary_Ungraded_Total,School_Name,School_No,School_Status,...,Year_12_Total,Year_1_Total,Year_2_Total,Year_3_Total,Year_4_Total,Year_5_Total,Year_6_Total,Year_7_Total,Year_8_Total,Year_9_Total
0,F,2022,2,316.0,47.0,316.0,0.0,St Monica's School,204,O,...,0.0,34.0,58.0,55.0,39.0,46.0,37.0,0.0,0.0,0.0
1,F,2022,2,351.0,52.0,351.0,0.0,St Joseph's School,205,O,...,0.0,55.0,42.0,66.0,46.0,46.0,44.0,0.0,0.0,0.0
2,F,2022,2,15.0,1.0,15.0,0.0,St Brendan's Primary School,212,O,...,0.0,2.0,3.0,3.0,0.0,3.0,3.0,0.0,0.0,0.0
3,F,2022,2,1463.0,0.0,0.0,0.0,Sacred Heart College,219,O,...,243.0,0.0,0.0,0.0,0.0,0.0,0.0,237.0,239.0,260.0
4,F,2022,2,110.0,17.0,110.0,0.0,St Patrick's School,227,O,...,0.0,23.0,21.0,11.0,14.0,10.0,14.0,0.0,0.0,0.0


Keeping only the shared columns makes the combined dataset more stable and avoids problems caused by differences between annual file formats.


## Check whether school numbers are unique enough

At this point, we wanted to confirm whether `School_No` could be used safely as a matching key.

A quick check shows that some school numbers appear more than once in the same year, which means they cannot always be treated as unique by themselves.


In [7]:
# Check counts by year
for year in sorted(enrolments_all_df["Census_Year"].unique()):
    df_year = enrolments_all_df[enrolments_all_df["Census_Year"] == year]
    print(
        f"Year {year}: Total rows = {len(df_year)}, "
        f"Unique School_No = {df_year['School_No'].nunique()}"
    )


Year 2022: Total rows = 2286, Unique School_No = 2145
Year 2023: Total rows = 2290, Unique School_No = 2149
Year 2024: Total rows = 2295, Unique School_No = 2155
Year 2025: Total rows = 2301, Unique School_No = 2162


In [8]:
# Show duplicate School_No examples
for year in sorted(enrolments_all_df["Census_Year"].unique()):
    df_year = enrolments_all_df[enrolments_all_df["Census_Year"] == year]
    duplicates = df_year[df_year.duplicated(subset=["School_No"], keep=False)]

    if not duplicates.empty:
        example_school = duplicates["School_No"].iloc[0]
        print(f"\nYear {year} - Example duplicate School_No: {example_school}")
        display(duplicates[duplicates["School_No"] == example_school].head())



Year 2022 - Example duplicate School_No: 250


,CENSUS_TYPE,Census_Year,Entity_Type,Grand_Total,Prep_Total,Primary_Total,Primary_Ungraded_Total,School_Name,School_No,School_Status,...,Year_12_Total,Year_1_Total,Year_2_Total,Year_3_Total,Year_4_Total,Year_5_Total,Year_6_Total,Year_7_Total,Year_8_Total,Year_9_Total
6,F,2022,2,1209.0,0.0,0.0,0.0,Star of the Sea College,250,O,...,194.0,0.0,0.0,0.0,0.0,0.0,0.0,203.0,206.0,198.0
492,F,2022,1,332.0,63.0,332.0,0.0,Flemington Primary School,250,O,...,0.0,60.0,61.0,36.0,55.0,26.0,31.0,0.0,0.0,0.0



Year 2023 - Example duplicate School_No: 26


,CENSUS_TYPE,Census_Year,Entity_Type,Grand_Total,Prep_Total,Primary_Total,Primary_Ungraded_Total,School_Name,School_No,School_Status,...,Year_12_Total,Year_1_Total,Year_2_Total,Year_3_Total,Year_4_Total,Year_5_Total,Year_6_Total,Year_7_Total,Year_8_Total,Year_9_Total
2288,F,2023,2,515.0,0.0,0.0,0.0,St Marys College Melbourne,26,O,...,94.0,0.0,0.0,0.0,0.0,0.0,0.0,73.0,95.0,72.0
2790,F,2023,1,278.0,42.0,278.0,0.0,Belmont Primary School,26,O,...,0.0,30.0,38.0,50.0,36.0,40.0,42.0,0.0,0.0,0.0



Year 2024 - Example duplicate School_No: 26


,CENSUS_TYPE,Census_Year,Entity_Type,Grand_Total,Prep_Total,Primary_Total,Primary_Ungraded_Total,School_Name,School_No,School_Status,...,Year_12_Total,Year_1_Total,Year_2_Total,Year_3_Total,Year_4_Total,Year_5_Total,Year_6_Total,Year_7_Total,Year_8_Total,Year_9_Total
4578,F,2024,2,474.0,0.0,0.0,0.0,St Marys College Melbourne,26,O,...,100.0,0.0,0.0,0.0,0.0,0.0,0.0,57.0,71.0,85.0
5076,F,2024,1,293.0,37.0,293.0,0.0,Belmont Primary School,26,O,...,0.0,47.0,29.0,44.0,53.0,40.0,43.0,0.0,0.0,0.0



Year 2025 - Example duplicate School_No: 1527


,CENSUS_TYPE,Census_Year,Entity_Type,Grand_Total,Prep_Total,Primary_Total,Primary_Ungraded_Total,School_Name,School_No,School_Status,...,Year_12_Total,Year_1_Total,Year_2_Total,Year_3_Total,Year_4_Total,Year_5_Total,Year_6_Total,Year_7_Total,Year_8_Total,Year_9_Total
6871,F,2025,2,161.4,24.0,161.4,0.0,St Margaret Mary's School,1527,O,...,0.0,26.0,32.4,15.0,33.0,11.0,20.0,0.0,0.0,0.0
7386,F,2025,1,15.0,3.0,15.0,0.0,Dookie Primary School,1527,O,...,0.0,1.0,2.0,1.0,1.0,5.0,2.0,0.0,0.0,0.0


Because of these repeated school numbers, merging on `School_No` alone would be risky.

To reduce that problem, a stronger identifier is needed before filtering and merging the datasets.


## Build a stronger school identifier

To make matching more reliable, a new identifier is created using both `Entity_Type` and `School_No`.

This gives a more consistent school key across the multi-year enrollment files and the location dataset.


In [9]:
def create_unique_id(row):
    if row["Entity_Type"] == 1:
        return 100000 + int(row["School_No"])
    elif row["Entity_Type"] == 2:
        return 200000 + int(row["School_No"])
    else:
        return int(row["School_No"])

enrolments_all_df["Unique_School_ID"] = enrolments_all_df.apply(create_unique_id, axis=1)

display(enrolments_all_df[["School_No", "Entity_Type", "Unique_School_ID"]].head(10))


,School_No,Entity_Type,Unique_School_ID
0,204,2,200204
1,205,2,200205
2,212,2,200212
3,219,2,200219
4,227,2,200227
5,236,2,200236
6,250,2,200250
7,251,2,200251
8,252,2,200252
9,262,2,200262


This extra identifier is mainly used to avoid incorrect matches when the same school number appears under different categories.


## Narrow the dataset for more consistent comparison

For the trend part of the project, we only want schools that can be followed properly across the full period.

So instead of using every row exactly as it appears, we reduce the dataset to schools that have a consistent presence across all four years. That makes later comparisons more meaningful.


In [10]:
# Review school status counts
status_counts = (
    enrolments_all_df.groupby(["Census_Year", "School_Status"])["Unique_School_ID"]
    .nunique()
    .reset_index(name="School_Count")
)

display(status_counts)


,Census_Year,School_Status,School_Count
0,2022,C,1
1,2022,O,2284
2,2022,U,1
3,2023,C,1
4,2023,O,2286
5,2023,U,3
6,2024,O,2294
7,2024,U,1
8,2025,C,4
9,2025,O,2297


In [11]:
# Limit the dataset to schools that are active in the comparison period
active_schools_df = enrolments_all_df[enrolments_all_df["School_Status"] == "O"].copy()

# Count number of years each school appears
school_year_counts = (
    active_schools_df.groupby("Unique_School_ID")["Census_Year"]
    .nunique()
    .reset_index(name="Year_Count")
)

schools_all_years = school_year_counts[school_year_counts["Year_Count"] == 4]["Unique_School_ID"]

consistent_schools_df = active_schools_df[
    active_schools_df["Unique_School_ID"].isin(schools_all_years)
].copy()

print("Schools included across all four years:")
print(consistent_schools_df.groupby("Census_Year")["Unique_School_ID"].nunique())


Schools included across all four years:
Census_Year
2022    2246
2023    2246
2024    2246
2025    2246
Name: Unique_School_ID, dtype: int64


This step is useful because it removes part of the noise that would come from schools appearing in only some years of the dataset.


## Prepare the location dataset for matching

Before merging the two datasets, the same school identifier needs to exist in the location data as well.

That way, the merge is based on the same key on both sides instead of relying on school number alone.


In [12]:
school_locations_df = school_locations_df.copy()
school_locations_df["School_No"] = school_locations_df["School_No"].astype(str).str.strip()
school_locations_df["Entity_Type"] = school_locations_df["Entity_Type"].astype(int)

school_locations_df["Unique_School_ID"] = school_locations_df.apply(create_unique_id, axis=1)

print("School location dataset with unique ID:")
display(school_locations_df[["School_No", "Entity_Type", "Unique_School_ID"]].head(10))


School location dataset with unique ID:


,School_No,Entity_Type,Unique_School_ID
0,20,2,200020
1,25,2,200025
2,26,2,200026
3,28,2,200028
4,29,2,200029
5,30,2,200030
6,33,2,200033
7,35,2,200035
8,60,2,200060
9,77,2,200077


## Merge enrollment and location information

Once the enrollment data and location data use the same identifier, they can be joined into one dataset.

This gives a table that includes both historical enrollment values and school profile information in the same place.


In [13]:
final_df = pd.merge(
    consistent_schools_df,
    school_locations_df,
    on="Unique_School_ID",
    how="left",
    suffixes=("_enrol", "_loc")
)

print("Final merged dataset shape:")
print(final_df.shape)
display(final_df.head())


Final merged dataset shape:
(8984, 50)


,CENSUS_TYPE,Census_Year,Entity_Type_enrol,Grand_Total,Prep_Total,Primary_Total,Primary_Ungraded_Total,School_Name_enrol,School_No_enrol,School_Status,...,Postal_Town,Postal_State,Postal_Postcode,Full_Phone_No,DE_Admin_Region,DE_Admin_AREA,LGA_ID,LGA_Name,X,Y
0,F,2022,2,316.0,47.0,316.0,0.0,St Monica's School,204,O,...,KANGAROO FLAT,VIC,3555,03 5447 7832,NORTH-WESTERN VICTORIA,Loddon Campaspe,262,Greater Bendigo (C),144.24,-36.79
1,F,2022,2,351.0,52.0,351.0,0.0,St Joseph's School,205,O,...,BENALLA,VIC,3672,03 5762 1347,NORTH-EASTERN VICTORIA,Ovens Murray,101,Benalla (RC),145.97,-36.55
2,F,2022,2,1463.0,0.0,0.0,0.0,Sacred Heart College,219,O,...,NEWTOWN,VIC,3220,03 5221 4211,SOUTH-WESTERN VICTORIA,Barwon,275,Greater Geelong (C),144.34,-38.14
3,F,2022,2,110.0,17.0,110.0,0.0,St Patrick's School,227,O,...,ST ARNAUD,VIC,3478,03 5495 1038,SOUTH-WESTERN VICTORIA,Wimmera South West,581,Northern Grampians (S),143.24,-36.61
4,F,2022,2,800.0,0.0,0.0,0.0,St Joseph's College Mildura,236,O,...,MILDURA,VIC,3500,03 5018 8000,NORTH-WESTERN VICTORIA,Mallee,478,Mildura (RC),142.15,-34.18


The merged dataset is the main working dataset for the next stage of the project.


## Run a few checks on the merged dataset

Before moving into deeper analysis, we checked the merged dataset for basic issues such as missing values, year coverage, and the number of unique schools included.

This is mainly to confirm that the merge works as expected.


In [14]:
print("Missing values in final dataset:")
display(final_df.isnull().sum().sort_values(ascending=False).head(15))


Missing values in final dataset:


,0
Address_Line_2,8944
Postal_Address_Line_2,8928
Y,4
X,4
Prep_Total,0
Primary_Total,0
Entity_Type_enrol,0
Grand_Total,0
School_No_enrol,0
School_Status,0


In [15]:
print("Unique schools in final dataset:", final_df["Unique_School_ID"].nunique())
print("Years included:", sorted(final_df["Census_Year"].unique()))
print("Education sectors:", final_df["Education_Sector"].dropna().unique()[:10])


Unique schools in final dataset: 2246
Years included: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Education sectors: ['Catholic' 'Government' 'Independent']
